# Module 12 — Thinking Budgets

The second Gemini-only unlock in Part 2. Simpler than grounding + caching; one knob, big effect.

Gemini 2.5+ models support **thinking** — an internal reasoning pass the model does before producing its final answer. The tokens consumed during thinking don't appear in the output text; they're billed as a separate category (`thoughts_token_count`) and they take real wall-clock time.

**`ThinkingConfig.thinking_budget`** is the knob. Set it to 0 for instant responses (no internal reasoning); set it to 2048 or higher for hard problems where reasoning pays off. The trade-off is explicit — you spend tokens and latency in exchange for answer quality on reasoning-heavy tasks.

**What you'll leave with:**
- Run the same problem at `thinking_budget=0` vs `thinking_budget=4096` and see the latency + thought-token counts.
- Understand when thinking earns its keep (complex multi-step reasoning, hard math, code debugging) and when it doesn't (factual lookup, simple questions).
- Wire `ThinkingConfig` into an ADK agent via `BuiltInPlanner`.

**Running cost:** under $0.01.

# Setup

In [ ]:
!pip install -q google-adk==1.28.0 google-genai litellm==1.83.4 python-dotenv==1.0.1 nest-asyncio==1.6.0 deprecated==1.2.18 2>/dev/null
print("✅ Packages installed.")

In [ ]:
import os, sys, warnings, time, uuid
warnings.filterwarnings("ignore")
try: sys.stderr.fileno()
except Exception: sys.stderr = open(os.devnull, "w")

GOOGLE_API_KEY = None
try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
except Exception:
    try:
        from dotenv import load_dotenv; load_dotenv()
        GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
    except ImportError: pass
if not GOOGLE_API_KEY:
    from getpass import getpass
    GOOGLE_API_KEY = getpass("Enter your Google AI Studio API key: ")
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"
print("✅ Environment ready.")

In [ ]:
import asyncio, logging
import nest_asyncio; nest_asyncio.apply()
logging.getLogger("LiteLLM").setLevel(logging.WARNING)

from google import genai
from google.genai import types
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.planners import BuiltInPlanner

print("✅ Imports successful.")

# How Thinking Works

When you give Gemini 2.5+ a thinking budget greater than zero, the model internally does a reasoning pass before producing its final answer. The internal reasoning is not returned in the `text` field. It's counted separately as `thoughts_token_count` in the usage metadata.

Two ways to control thinking:

| Setting | Type | Meaning |
|---|---|---|
| `thinking_budget=0` | int | Disable thinking. Instant response. |
| `thinking_budget=N` | int | Allow up to N thought-tokens (higher = more reasoning) |
| `thinking_level=LOW/MEDIUM/HIGH` | enum | Gemini 3+ only; coarser preset |

For this notebook we use `thinking_budget` (int) — works on Gemini 2.5.

# Demo 1 — Direct google-genai A/B Comparison

Same problem at `thinking_budget=0` (no reasoning) and `thinking_budget=4096` (up to 4K reasoning tokens). Same model. Compare latency and answer quality.

In [ ]:
client = genai.Client(api_key=GOOGLE_API_KEY)

# A reasoning-heavy problem — multiple steps, easy to get wrong.
PROBLEM = """A small bakery sells three types of bread:
- Sourdough: 4.50 EUR, takes 30 minutes labor
- Rye: 3.80 EUR, takes 25 minutes labor
- Whole wheat: 4.10 EUR, takes 20 minutes labor

Flour costs are:
- Sourdough: 1.20 EUR per loaf
- Rye: 1.00 EUR per loaf
- Whole wheat: 0.90 EUR per loaf

Labor costs 0.40 EUR per minute (including overhead).

Which bread has the highest profit margin per loaf in percentage terms (profit / revenue × 100)?
Show your calculation for each and state the winner."""

for budget in [0, 4096]:
    print(f"\n── thinking_budget={budget} ──")
    t0 = time.time()
    resp = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=PROBLEM,
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(
                thinking_budget=budget,
                include_thoughts=False,
            ),
        ),
    )
    elapsed = time.time() - t0
    usage = resp.usage_metadata
    thoughts = getattr(usage, "thoughts_token_count", None) or 0
    print(f"latency: {elapsed:.2f}s")
    print(f"tokens — input: {usage.prompt_token_count}, thoughts: {thoughts}, output: {usage.candidates_token_count}")
    print(f"answer:\n{resp.text}")
    print()

Read the two outputs carefully. Three things to compare:

1. **Latency.** budget=0 is well under a second; budget=4096 takes several seconds. The difference is real reasoning time the model is spending internally.
2. **Thought tokens.** budget=0 uses 0 (or near-0) thought tokens. budget=4096 uses however many the model decided the problem needed — often less than the budget cap; that's the budget behaving as a ceiling, not a target.
3. **Answer quality.** On a multi-step numerical problem like this, `thinking_budget=4096` produces a cleaner, better-organized, usually more accurate answer. budget=0 sometimes skips steps, mis-does arithmetic, or picks the wrong winner.

The cost model: you pay for thought tokens the same rate as output tokens. 527 thought tokens at Flash's $0.30/M = ~$0.00016 per invocation. Worth it for hard problems; wasted for easy ones.

# Demo 2 — Thinking Budgets Inside an ADK Agent

`BuiltInPlanner` wraps a `ThinkingConfig` so you can apply it to an `LlmAgent`. Same API; ADK plumbs it through to the underlying Gemini call.

In [ ]:
# An agent that uses thinking explicitly
thinking_agent = LlmAgent(
    name="thinking_agent",
    model="gemini-2.5-flash",
    description="Reasoning agent with a generous thinking budget.",
    instruction="Answer user questions. For multi-step problems, reason carefully.",
    planner=BuiltInPlanner(
        thinking_config=types.ThinkingConfig(thinking_budget=2048),
    ),
)

# A counterpart with thinking disabled
fast_agent = LlmAgent(
    name="fast_agent",
    model="gemini-2.5-flash",
    description="Fast agent; no internal reasoning.",
    instruction="Answer user questions concisely.",
    planner=BuiltInPlanner(
        thinking_config=types.ThinkingConfig(thinking_budget=0),
    ),
)

APP = "m12"
USER = "student"
session_service = InMemorySessionService()

async def ask_agent(agent, prompt: str, label: str):
    sid = f"s-{uuid.uuid4().hex[:6]}"
    await session_service.create_session(app_name=APP, user_id=USER, session_id=sid)
    runner = Runner(agent=agent, app_name=APP, session_service=session_service)
    msg = types.Content(role="user", parts=[types.Part(text=prompt)])
    t0 = time.time()
    final = ""
    async for ev in runner.run_async(user_id=USER, session_id=sid, new_message=msg):
        if ev.is_final_response() and ev.content and ev.content.parts:
            for p in ev.content.parts:
                if p.text:
                    final = p.text.strip()
    print(f"── {label} ({time.time() - t0:.2f}s) ──")
    print(final[:400])
    print()

QUESTION = "What is the sum of all prime numbers between 20 and 50?"

await ask_agent(fast_agent,     QUESTION, "fast_agent (thinking=0)")
await ask_agent(thinking_agent, QUESTION, "thinking_agent (thinking=2048)")

For a problem like this — requiring you to enumerate primes and sum them — `thinking_agent` tends to get the correct answer (23 + 29 + 31 + 37 + 41 + 43 + 47 = 251). `fast_agent` sometimes misses a prime, or makes an arithmetic error. The quality difference on reasoning-heavy tasks is real.

For simple factual queries (*"What's the capital of France?"*), thinking adds latency with no benefit. The knob should be turned high for hard problems and low for easy ones. Per-agent is the reasonable granularity — a single coordinator with specialist sub-agents can route hard queries to a high-thinking specialist and trivia to a fast one.

# When Thinking Earns Its Keep

Thinking budgets are not free. Tokens cost money. Latency costs user patience. Crank the budget for:

- **Multi-step reasoning** — the example problems above. Numerical reasoning, logic puzzles, scheduling.
- **Code debugging** — the model has to trace logic; thinking helps significantly.
- **Hard math** — arithmetic, algebra, proofs.
- **Multi-constraint planning** — itinerary-building, resource allocation, dependency-aware scheduling.

Skip thinking for:

- **Factual lookups** — "capital of France", "weather in Prague". The model already knows; no reasoning needed.
- **Text transformations** — summarization, translation, style rewriting.
- **Simple classification** — "is this email spam?" Usually pattern-matching, not reasoning.

A good production heuristic: **route on query type**. A router agent (cheap, fast, no thinking) inspects the user's question and delegates to either a thinking-heavy specialist or a fast specialist. We built exactly this composition pattern in M06.

# Gemini 3+ — Thinking Levels

On Gemini 3+, the API gains a coarser-grained control:

| Setting | Maps to roughly |
|---|---|
| `thinking_level=MINIMAL` | Off / very low budget |
| `thinking_level=LOW` | ~1K thoughts |
| `thinking_level=MEDIUM` | ~4K thoughts |
| `thinking_level=HIGH` | Unlimited (model decides) |

Use `thinking_level` on Gemini 3+ when you don't want to tune a number; use `thinking_budget` on 2.5 (and 3+) when you want precise control.

One caveat worth knowing: **thought signatures persist across multi-turn tool calls on Gemini** — which means the thinking budget can be *reused* mid-conversation without reloading the reasoning. This is a Gemini-only property; no other frontier model ships it as of April 2026.

# LiteLLM Parity — Partial

Other providers ship something similar, with different vocabularies:

- **OpenAI GPT-5 / o3**: `reasoning_effort` parameter (low/medium/high). Passes through LiteLLM.
- **Anthropic Claude 4.5+**: extended thinking via `thinking={"type": "enabled", "budget_tokens": N}`. Partial through LiteLLM — works for basic reasoning, breaks on tool calls.
- **Qwen / DeepSeek**: reasoning variants (e.g. `qwen3-32b-thinking`) have reasoning on by default; some support a `reasoning.effort` parameter.

**None are as clean as Gemini's `ThinkingConfig`.** If reasoning budgets are a first-class requirement for your agent, native Gemini is where you get the most control.

# Your Turn

1. **A problem where thinking matters.** Pose a 4-step logic puzzle to both `fast_agent` and `thinking_agent`. Does the fast one miss steps? Which gets the right answer?
2. **A problem where thinking doesn't matter.** Ask both agents for the capital of France. Is the latency difference noticeable? Is the answer the same?
3. **Scale the budget.** Create agents with budgets at 0, 512, 2048, 8192. Ask them a hard math problem. At what budget does quality stop improving?
4. **Include thoughts.** Set `include_thoughts=True` in a ThinkingConfig and check the response for thought tokens. What does the model's internal reasoning look like?

# Key Takeaways

- **Gemini 2.5+ supports `ThinkingConfig`** — a knob that allocates up to N internal reasoning tokens before the final answer.
- **`thinking_budget=0`** disables reasoning (instant, maybe wrong on hard problems). **`thinking_budget=2048+`** enables real reasoning (slower, more accurate on hard problems).
- **Cost model:** thought tokens billed at the output-token rate. Modest per-invocation cost; worth it for hard problems.
- **`BuiltInPlanner`** plumbs `ThinkingConfig` through to an `LlmAgent`. Same API from ADK's perspective.
- **Gemini 3+ adds `thinking_level`** (MINIMAL / LOW / MEDIUM / HIGH) — a coarser preset for when you don't want to tune a number.
- **Thought signatures persist across multi-turn tool calls** — Gemini-only. Reasoning carries through without reload.
- **Production pattern:** router decides between a thinking-heavy agent and a fast agent per query.
- **LiteLLM parity is partial.** OpenAI has `reasoning_effort`; Claude has `thinking=...`; Qwen has reasoning variants. None as clean as Gemini's `ThinkingConfig`.

# Next up — M13: Live API voice agent

The third Gemini unlock. Bidirectional audio streaming — voice-in, voice-out — with voice activity detection and interruption handling. The most differentiated Gemini-only capability on the market as of April 2026. Heads up: the Live API is genuinely fragile in some environments. If the demo doesn't run end-to-end on your machine, see `DEMOS_BROKEN.md` for the fallback.